# CUM Optimizer — Deep Research Benchmark Suite

**Run on Colab with H100 GPU**

Tests all 9 untested directions from deep research v1 against our current best (5v6 SVD ns_blend).

| # | Name | Direction | Key Idea |
|---|------|-----------|----------|
| 6v1 | Polar Express | D1 | Remez-optimal per-step NS coefficients |
| 6v2 | PolarGrad | D4 | Nuclear norm scaling for null-gradient consistency |
| 6v3 | PSGD Kron | D5 | Learned Kronecker preconditioner |
| 6v4 | Dion | D6 | Amortized power iteration + QR + small SVD |
| 6v5 | Zolotarev | D7 | Rational approximation (Halley/QDWH) |
| 6v6 | MARS | D8 | Variance reduction wrapping Muon |
| 6v7 | Warm-Started NS | D9 | Previous polar factor as NS starting point |
| 6v8 | Optimal SV Shrinkage | D10 | Donoho-Gavish denoising of gradient SVs |
| 6v9 | Weighted Procrustes | D11 | Direction-aware orthogonal optimizer |

In [ ]:
# Clone repo and setup path
import os, subprocess, sys

REPO_URL = 'https://github.com/BrownHujay/Curvature-Unified-Muon.git'
REPO_DIR = '/content/Curvature-Unified-Muon'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
    print(f'Cloned repo to {REPO_DIR}')
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
    print(f'Repo already exists, pulled latest')

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print(f'Working directory: {os.getcwd()}')

# Verify GPU
import torch
print(f'\nPyTorch {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
    DEVICE = torch.device('cuda')
else:
    print('No GPU — running on CPU (will be slow)')
    DEVICE = torch.device('cpu')

In [ ]:
# Download TinyShakespeare if not present
import os
DATA_DIR = 'benchmarks/tier0/data'
DATA_PATH = os.path.join(DATA_DIR, 'tinyshakespeare.txt')

if not os.path.exists(DATA_PATH):
    os.makedirs(DATA_DIR, exist_ok=True)
    import urllib.request
    url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
    print(f'Downloading TinyShakespeare...')
    urllib.request.urlretrieve(url, DATA_PATH)

with open(DATA_PATH, 'r') as f:
    TEXT = f.read()
print(f'Dataset: {len(TEXT):,} chars')

In [ ]:
# Core imports
import sys, time, math
import numpy as np

from benchmarks.models.micro_gpt import MicroGPT
from cum.newton_schulz import newton_schulz_orthogonalize
from cum.utils import aspect_ratio_scale
from cum import (
    CUM5v6, CUM5v7,
    CUM6v1, CUM6v2, CUM6v3, CUM6v4, CUM6v5, CUM6v6, CUM6v7, CUM6v8, CUM6v9,
)
print('All optimizers imported OK')

In [ ]:
# ── Data & Training Infrastructure ──

class CharDataset:
    def __init__(self, text, ctx_len=256, device='cpu'):
        self.chars = sorted(set(text))
        self.char_to_idx = {c: i for i, c in enumerate(self.chars)}
        self.vocab_size = len(self.chars)
        self.data = torch.tensor([self.char_to_idx[c] for c in text], dtype=torch.long, device=device)
        self.ctx_len = ctx_len

    def get_batch(self, batch_size, split='train', train_ratio=0.9):
        n = int(len(self.data) * train_ratio)
        data = self.data[:n] if split == 'train' else self.data[n:]
        ix = torch.randint(0, len(data) - self.ctx_len - 1, (batch_size,), device=self.data.device)
        x = torch.stack([data[i:i+self.ctx_len] for i in ix])
        y = torch.stack([data[i+1:i+self.ctx_len+1] for i in ix])
        return x, y


class SimpleMuon(torch.optim.Optimizer):
    """Baseline Muon optimizer."""
    def __init__(self, params, lr=0.02, beta1=0.95, ns_steps=5):
        defaults = dict(lr=lr, beta1=beta1, ns_steps=ns_steps)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None:
                    continue
                state = self.state[p]
                if len(state) == 0:
                    state['momentum_buffer'] = torch.zeros_like(p)
                m = state['momentum_buffer']
                m.mul_(group['beta1']).add_(p.grad, alpha=1 - group['beta1'])
                u = p.grad + group['beta1'] * m
                orth = newton_schulz_orthogonalize(u, steps=group['ns_steps'])
                scale = math.sqrt(max(1, p.shape[0] / p.shape[1]))
                p.add_(orth, alpha=-group['lr'] * scale)


def get_lr_schedule(step, total_steps, warmup_steps, base_lr):
    if step < warmup_steps:
        return base_lr * step / max(1, warmup_steps)
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return base_lr * 0.5 * (1 + math.cos(math.pi * progress))


print('Infrastructure ready')

In [ ]:
# ── Training Loop ──

def train_one(name, main_opt, aux_opt, model, dataset,
              total_steps=2000, batch_size=32, warmup_steps=200,
              base_lr=0.02, eval_every=500):
    """Train one config and return (final_val_loss, trajectory, elapsed)."""
    model.train()
    trajectory = []
    t0 = time.perf_counter()

    # Warmup compiled model
    for _ in range(10):
        x, y = dataset.get_batch(batch_size, split='train')
        _, _ = model(x, y)

    for step in range(total_steps):
        current_lr = get_lr_schedule(step, total_steps, warmup_steps, base_lr)
        if main_opt:
            for g in main_opt.param_groups:
                g['lr'] = current_lr

        x, y = dataset.get_batch(batch_size, split='train')
        _, loss = model(x, y)

        if main_opt:
            main_opt.zero_grad()
        aux_opt.zero_grad()
        loss.backward()
        if main_opt:
            main_opt.step()
        aux_opt.step()

        if step % eval_every == 0 or step == total_steps - 1:
            model.eval()
            val_losses = []
            for _ in range(10):
                vx, vy = dataset.get_batch(batch_size, split='val')
                with torch.no_grad():
                    _, vl = model(vx, vy)
                    val_losses.append(vl.item())
            val_loss = np.mean(val_losses)
            trajectory.append((step, val_loss))
            model.train()
            elapsed = time.perf_counter() - t0
            print(f'  [{name}] step {step}: val_loss={val_loss:.4f} ({elapsed:.0f}s)')

    elapsed = time.perf_counter() - t0
    model.eval()
    val_losses = []
    for _ in range(20):
        vx, vy = dataset.get_batch(batch_size, split='val')
        with torch.no_grad():
            _, vl = model(vx, vy)
            val_losses.append(vl.item())
    final_val = np.mean(val_losses)
    return final_val, trajectory, elapsed


print('Training loop ready')

In [ ]:
# ── Optimizer Factory ──

def make_model_and_opts(dataset, cfg, device=DEVICE):
    """Create model + optimizer pair from config dict."""
    torch.manual_seed(42)
    model = MicroGPT(
        vocab_size=dataset.vocab_size,
        d_model=128, n_heads=4, n_layers=4, d_ff=512, ctx_len=256,
    ).to(device)

    # torch.compile for GPU
    if device.type == 'cuda':
        model = torch.compile(model, mode='reduce-overhead')

    hidden_2d = model.get_hidden_2d_params()
    other = model.get_other_params()
    aux_opt = torch.optim.AdamW(other, lr=3e-4, weight_decay=0.01)

    opt_type = cfg['type']

    if opt_type == 'Muon':
        main_opt = SimpleMuon(hidden_2d, lr=0.02, ns_steps=5)

    # ── Current best ──
    elif opt_type == '5v6':
        main_opt = CUM5v6(
            hidden_2d, lr=0.02,
            mode=cfg.get('mode', 'ns_blend'),
            ns_save_at=cfg.get('ns_save_at', 2),
            ns_blend=cfg.get('ns_blend', 0.25),
        )
    elif opt_type == '5v7':
        main_opt = CUM5v7(
            hidden_2d, lr=0.02,
            mode=cfg.get('mode', 'top_preserve'),
            ns_top_steps=cfg.get('ns_top_steps', 3),
            ns_bot_steps=cfg.get('ns_bot_steps', 5),
            split_frac=cfg.get('split_frac', 0.25),
            blend_start=cfg.get('blend_start', 0.35),
            blend_end=cfg.get('blend_end', 0.10),
        )

    # ── Deep Research Direction 1: Polar Express ──
    elif opt_type == '6v1':
        main_opt = CUM6v1(
            hidden_2d, lr=0.02,
            mode=cfg.get('mode', 'standard'),
            ns_save_at=cfg.get('ns_save_at', 2),
            ns_blend=cfg.get('ns_blend', 0.15),
        )

    # ── Deep Research Direction 4: PolarGrad ──
    elif opt_type == '6v2':
        main_opt = CUM6v2(
            hidden_2d, lr=0.02,
            mode=cfg.get('mode', 'ns'),
            ns_steps=cfg.get('ns_steps', 5),
            norm_scale=cfg.get('norm_scale', 1.0),
            ns_blend=cfg.get('ns_blend', 0.15),
            ns_save_at=cfg.get('ns_save_at', 2),
        )

    # ── Deep Research Direction 5: PSGD Kron ──
    elif opt_type == '6v3':
        main_opt = CUM6v3(
            hidden_2d, lr=0.02,
            precond_lr=cfg.get('precond_lr', 0.1),
            precond_freq=cfg.get('precond_freq', 10),
        )

    # ── Deep Research Direction 6: Dion ──
    elif opt_type == '6v4':
        main_opt = CUM6v4(
            hidden_2d, lr=0.02,
            rank=cfg.get('rank', None),  # None = full rank
            power_iters=cfg.get('power_iters', 2),
        )

    # ── Deep Research Direction 7: Zolotarev/Halley ──
    elif opt_type == '6v5':
        main_opt = CUM6v5(
            hidden_2d, lr=0.02,
            mode=cfg.get('mode', 'halley'),
            iters=cfg.get('iters', 3),
        )

    # ── Deep Research Direction 8: MARS ──
    elif opt_type == '6v6':
        main_opt = CUM6v6(
            hidden_2d, lr=0.02,
            gamma=cfg.get('gamma', 0.5),
            ns_steps=cfg.get('ns_steps', 5),
        )

    # ── Deep Research Direction 9: Warm-Started NS ──
    elif opt_type == '6v7':
        main_opt = CUM6v7(
            hidden_2d, lr=0.02,
            mode=cfg.get('mode', 'warm2'),
            warm_steps=cfg.get('warm_steps', 2),
            ns_blend=cfg.get('ns_blend', 0.15),
        )

    # ── Deep Research Direction 10: Optimal SV Shrinkage ──
    elif opt_type == '6v8':
        main_opt = CUM6v8(
            hidden_2d, lr=0.02,
            mode=cfg.get('mode', 'hard'),
            noise_est=cfg.get('noise_est', 'median'),
            ns_blend=cfg.get('ns_blend', 0.5),
        )

    # ── Deep Research Direction 11: Weighted Procrustes ──
    elif opt_type == '6v9':
        main_opt = CUM6v9(
            hidden_2d, lr=0.02,
            mode=cfg.get('mode', 'magnitude'),
            alpha=cfg.get('alpha', 0.5),
            decay=cfg.get('decay', 1.0),
            adapt_lr=cfg.get('adapt_lr', 0.01),
        )

    else:
        raise ValueError(f'Unknown optimizer type: {opt_type}')

    return model, main_opt, aux_opt


print('Factory ready')

In [ ]:
# ── Benchmark Runner ──

def run_benchmark(configs, dataset, total_steps=2000, eval_every=500, device=DEVICE):
    """Run all configs and return results dict."""
    results = []
    for i, cfg in enumerate(configs):
        name = cfg['name']
        print(f'\n{"="*60}')
        print(f'[{i+1}/{len(configs)}] {name}')
        print(f'{"="*60}')

        try:
            model, main_opt, aux_opt = make_model_and_opts(dataset, cfg, device)
            val_loss, trajectory, elapsed = train_one(
                name, main_opt, aux_opt, model, dataset,
                total_steps=total_steps, eval_every=eval_every,
            )
            results.append({
                'name': name, 'type': cfg['type'], 'val_loss': val_loss,
                'trajectory': trajectory, 'elapsed': elapsed, 'error': None,
            })
            print(f'  FINAL: {name} → val_loss={val_loss:.4f} ({elapsed:.1f}s)')
        except Exception as e:
            print(f'  ERROR: {name} → {e}')
            results.append({
                'name': name, 'type': cfg['type'], 'val_loss': float('inf'),
                'trajectory': [], 'elapsed': 0, 'error': str(e),
            })

        # Clear GPU memory between runs
        if device.type == 'cuda':
            torch.cuda.empty_cache()

    return results


def print_summary(results):
    """Print results table sorted by val_loss."""
    print(f'\n{"="*70}')
    print('RESULTS SUMMARY')
    print(f'{"="*70}')

    # Find baselines
    muon_loss = next((r['val_loss'] for r in results if r['type'] == 'Muon'), None)
    best5v6_loss = next((r['val_loss'] for r in results if r['type'] == '5v6'), None)

    sorted_results = sorted(results, key=lambda r: r['val_loss'])

    print(f'{"Name":30s} {"Val Loss":>10s} {"vs Muon":>10s} {"vs 5v6":>10s} {"Time":>8s} {"Status":>10s}')
    print('-' * 80)
    for r in sorted_results:
        if r['error']:
            print(f'{r["name"]:30s} {"FAILED":>10s} {"":>10s} {"":>10s} {"":>8s} {r["error"][:20]:>10s}')
            continue
        vs_muon = f'{r["val_loss"] - muon_loss:+.4f}' if muon_loss else 'N/A'
        vs_5v6 = f'{r["val_loss"] - best5v6_loss:+.4f}' if best5v6_loss else 'N/A'
        print(f'{r["name"]:30s} {r["val_loss"]:10.4f} {vs_muon:>10s} {vs_5v6:>10s} {r["elapsed"]:7.1f}s')

    # Highlight best new result
    new_results = [r for r in sorted_results if r['type'] not in ('Muon', '5v6') and not r['error']]
    if new_results:
        best = new_results[0]
        print(f'\n★ BEST NEW: {best["name"]} → {best["val_loss"]:.4f}', end='')
        if muon_loss:
            print(f' (vs Muon: {best["val_loss"] - muon_loss:+.4f})', end='')
        if best5v6_loss:
            print(f' (vs 5v6: {best["val_loss"] - best5v6_loss:+.4f})', end='')
        print()


print('Runner ready')

## Series 6a: All Deep Research Directions (Best Configs)

First pass — run one config per direction to find which are promising.

In [ ]:
# ── Series 6a: One config per direction ──
dataset = CharDataset(TEXT, ctx_len=256, device=DEVICE)
print(f'Dataset on {DEVICE}: {len(TEXT):,} chars, vocab={dataset.vocab_size}')

CONFIGS_6A = [
    # Baselines
    {'name': 'Muon NS=5',          'type': 'Muon'},
    {'name': '5v6 best (s2 b=0.25)', 'type': '5v6', 'mode': 'ns_blend', 'ns_save_at': 2, 'ns_blend': 0.25},

    # D1: Polar Express
    {'name': '6v1 PolarExpress std', 'type': '6v1', 'mode': 'standard'},
    {'name': '6v1 PolarExpress blend', 'type': '6v1', 'mode': 'blend', 'ns_save_at': 2, 'ns_blend': 0.15},

    # D4: PolarGrad
    {'name': '6v2 PolarGrad NS',    'type': '6v2', 'mode': 'ns'},
    {'name': '6v2 PolarGrad blend', 'type': '6v2', 'mode': 'ns_blend', 'ns_blend': 0.15, 'ns_save_at': 2},

    # D5: PSGD Kron
    {'name': '6v3 PSGD Kron',       'type': '6v3', 'precond_lr': 0.1, 'precond_freq': 10},

    # D6: Dion (full rank = exact polar)
    {'name': '6v4 Dion full',       'type': '6v4', 'rank': None, 'power_iters': 2},
    {'name': '6v4 Dion r=32',       'type': '6v4', 'rank': 32, 'power_iters': 2},

    # D7: Zolotarev/Halley
    {'name': '6v5 Halley 3iter',    'type': '6v5', 'mode': 'halley', 'iters': 3},
    {'name': '6v5 QDWH 3iter',      'type': '6v5', 'mode': 'qdwh', 'iters': 3},

    # D8: MARS
    {'name': '6v6 MARS γ=0.5',     'type': '6v6', 'gamma': 0.5},
    {'name': '6v6 MARS γ=1.0',     'type': '6v6', 'gamma': 1.0},

    # D9: Warm-Started NS
    {'name': '6v7 WarmNS 2step',    'type': '6v7', 'mode': 'warm2', 'warm_steps': 2},
    {'name': '6v7 WarmNS hybrid',   'type': '6v7', 'mode': 'hybrid', 'warm_steps': 2, 'ns_blend': 0.15},

    # D10: Optimal SV Shrinkage
    {'name': '6v8 Shrink hard',     'type': '6v8', 'mode': 'hard'},
    {'name': '6v8 Shrink blend',    'type': '6v8', 'mode': 'blend', 'ns_blend': 0.5},

    # D11: Weighted Procrustes
    {'name': '6v9 WProc mag a=0.5', 'type': '6v9', 'mode': 'magnitude', 'alpha': 0.5},
    {'name': '6v9 WProc decay',     'type': '6v9', 'mode': 'rank_decay', 'decay': 1.0},
]

print(f'Will run {len(CONFIGS_6A)} configs')

In [ ]:
# RUN Series 6a
results_6a = run_benchmark(CONFIGS_6A, dataset, total_steps=2000, eval_every=500)
print_summary(results_6a)

## Series 6b: Sweep best performers

After 6a, pick the top 2-3 directions and sweep their hyperparameters.

**Edit the configs below based on 6a results.**

In [ ]:
# ── Series 6b: Hyperparameter sweep for top performers ──
# EDIT THESE based on 6a results!

CONFIGS_6B = [
    # Baselines (always include)
    {'name': 'Muon NS=5',           'type': 'Muon'},
    {'name': '5v6 best (s2 b=0.25)', 'type': '5v6', 'mode': 'ns_blend', 'ns_save_at': 2, 'ns_blend': 0.25},

    # TODO: Add sweep configs for top performers from 6a
    # Example sweeps:
    # {'name': '6v1 blend b=0.10', 'type': '6v1', 'mode': 'blend', 'ns_blend': 0.10},
    # {'name': '6v1 blend b=0.20', 'type': '6v1', 'mode': 'blend', 'ns_blend': 0.20},
    # {'name': '6v1 blend b=0.25', 'type': '6v1', 'mode': 'blend', 'ns_blend': 0.25},
]

if len(CONFIGS_6B) > 2:
    results_6b = run_benchmark(CONFIGS_6B, dataset, total_steps=2000, eval_every=500)
    print_summary(results_6b)
else:
    print('Add sweep configs above based on 6a results, then re-run this cell')

## Series 6c: Combinations

Test combinations of the best ideas (e.g., MARS + Polar Express, PolarGrad + SVD blend, etc.)

In [ ]:
# ── Series 6c: Combinations ──
# EDIT based on 6a/6b results

CONFIGS_6C = [
    {'name': 'Muon NS=5',           'type': 'Muon'},
    {'name': '5v6 best (s2 b=0.25)', 'type': '5v6', 'mode': 'ns_blend', 'ns_save_at': 2, 'ns_blend': 0.25},

    # TODO: Add combination configs
    # e.g., if MARS and Polar Express both helped:
    # Implement a combined optimizer and test here
]

if len(CONFIGS_6C) > 2:
    results_6c = run_benchmark(CONFIGS_6C, dataset, total_steps=2000, eval_every=500)
    print_summary(results_6c)
else:
    print('Add combination configs above, then re-run this cell')

## Quick Results Copy-Paste

Run this cell to get a markdown table you can paste back to Claude.

In [ ]:
# Generate markdown results table for pasting back
def results_to_markdown(results, series_name='6a'):
    muon_loss = next((r['val_loss'] for r in results if r['type'] == 'Muon'), None)
    best5v6 = next((r['val_loss'] for r in results if r['type'] == '5v6'), None)

    lines = [f'## Series {series_name} Results\n']
    lines.append(f'| Config | Val Loss | vs Muon | vs 5v6 | Time |')
    lines.append(f'|--------|----------|---------|--------|------|')

    for r in sorted(results, key=lambda x: x['val_loss']):
        if r['error']:
            lines.append(f'| {r["name"]} | FAILED | — | — | — | {r["error"][:30]} |')
            continue
        vs_m = f'{r["val_loss"] - muon_loss:+.4f}' if muon_loss else '—'
        vs_5 = f'{r["val_loss"] - best5v6:+.4f}' if best5v6 else '—'
        lines.append(f'| {r["name"]} | {r["val_loss"]:.4f} | {vs_m} | {vs_5} | {r["elapsed"]:.0f}s |')

    md = '\n'.join(lines)
    print(md)
    return md

# Print results for all completed series
if 'results_6a' in dir():
    results_to_markdown(results_6a, '6a')
if 'results_6b' in dir() and len(CONFIGS_6B) > 2:
    print()
    results_to_markdown(results_6b, '6b')